# ICT-36h — F-Lens x Random Forest : le decodeur du papier LSD et l'honnetete de sa validation croisee (jumeau C#)

[Index ICT-Series](README.md) | [<< ICT-36g](ICT-36g-FLens-InteractionTransmodal-Python.ipynb) | [>> ICT-37](ICT-37-FLens-BeliefState-Python.ipynb) | [Version Python](ICT-36h-FLens-RFNestedCV-Python.ipynb)

**Jumeau C# (BCL)** du carnet [Python](ICT-36h-FLens-RFNestedCV-Python.ipynb). Moteur : **.NET Interactive (C#)**, uniquement la BCL — pas de NuGet, pas de reseau, pas de scikit-learn. Les deux descripteurs fractaux (HFD, Lempel-Ziv) sont des **copies declarees** des organes de reference *antropy 0.1.6* ; la foret aleatoire, les plis groupes et la validation croisee imbriquee sont des **reimplantations BCL** de la meme procedure.

> **Parite des jumeaux.** La zone d'egalite exacte — 204 echantillons, 17 sujets, sujets communs 0 (plis groupes) / 16-17 (plis non groupes), somme des importances = 1, plancher de p = 1/31 = 0,032, ancrages descripteurs — est reproduite par le jumeau. Les **precisions CV** diffèrent d'un jumeau a l'autre : le tirage `System.Random` (C#) n'est pas identique a `numpy.default_rng` (Python), et ce n'est **pas** un ecart a combler. La zone d'egalite porte la **structure** et les **verdicts qualitatifs** (Q1, Q3, Q4), pas les valeurs aleatoires.

Les figures matplotlib du jumeau Python sont remplacees par des **tableaux imprimes** (BCL sans pyplot).

**Objectifs** — ce carnet monte le dernier instrument nomme du papier de Subramani et al. (2026) encore absent de la serie : le **classifieur a foret aleatoire** qui decode la condition (LSD contre placebo) a partir de descripteurs fractals, sous **validation croisee imbriquee et groupee**. Puis il le calibre a verite terrain, et mesure trois gestes de validation qui decident de la valeur d'une precision rapportee :

1. la recuperation de la condition, contre une **nulle par permutation des etiquettes** ;
2. le **biais optimiste** d'une validation croisee non imbriquee ;
3. la **fuite de groupe** quand les plis ignorent le sujet ;
4. le **seuil de puissance** : a quelle amplitude d'effet la recuperation sort-elle de la nulle ?


## 1. Mise en place

Le papier (Subramani, Nest et al. 2026, bioRxiv, section 2.10 p. 11) decode la condition avec une **foret aleatoire** entrainee sur des descripteurs extraits des signaux MEG : puissance par bande, frequences de pic, complexite de Lempel-Ziv (LZC) et dimension fractale de Higuchi (HFD). La validation est une **validation croisee imbriquee** (K exterieur = 4, K interieur = 3) : la boucle interne choisit les hyperparametres, la boucle externe mesure sur des plis jamais vus. La significativite vient d'une **nulle par permutation des etiquettes**, qui donne la distribution de la precision sous l'hypothese nulle.

Le terrain de la serie ne porte pas la puissance spectrale (c'est l'objet d'ICT-36f) : on decode ici sur les **deux descripteurs fractals**, HFD et LZC.

### Questions pre-enregistrees

| # | Question | Prediction pre-enregistree | Seuil falsifiable |
|---|---|---|---|
| Q1 | Le decodeur recupere-t-il la condition a partir de HFD et LZC ? | oui, au-dessus de la nulle | precision multi-domaine >= 0,65 **et** > quantile 95 de la nulle |
| Q2 | Une validation non imbriquee surestime-t-elle la precision ? | biais >= 0, et il croit avec la finesse de la grille | biais(grille fine) >= biais(grille grossiere) >= 0 ; biais > 0,05 = surestimation materielle |
| Q3 | Ignorer le sujet dans les plis gonfle-t-il la precision ? | oui | ecart (plis non groupes - plis groupes) >= +0,03 |
| Q4 | La recuperation franchit-elle la bande nulle a une amplitude d'effet mesurable ? | oui, croissance monotone | monotonie de la precision en eta, franchissement entre eta = 0 et eta = 0,06 |

**Calibrage declare.** Avant l'execution, le meme generateur a ete joue hors carnet sur une seule graine pour choisir l'amplitude de travail `ETA_DECODE = 0,06` : a cette valeur la precision multi-domaine vaut environ 0,73 (ni plancher ni plafond). Les predictions ci-dessus sont fixees avant la mesure a quatre graines.


In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// === Jumeau C# (BCL) -- bibliotheque du carnet ===
// Descripteurs : copies declarees d'antropy 0.1.6 (HFD = formulation mne-features,
// LZC = Lempel-Ziv de Naereen). Foret, plis et CV : reimplantation BCL de la
// procedure du jumeau Python (pas de scikit-learn, pas de NuGet, pas de reseau).
public static class Iit
{
    // constantes (posees en fin de cellule)
    public static int[] SEEDS;
    public static int N_SUBJ, N_ROI, EPOCHS_PER_SUBJ, T_EPOCH, KMAX_HFD, N_TREES, N_PERM;
    public static double RHO_PLA, SIGMA_RHO_SUBJ, SIGMA_HFD_SUBJ, SIGMA_LZC_SUBJ;

    // assertion native : une violation est un TROUVE, jamais un silence
    public static void Check(bool cond, string msg)
    { if (!cond) throw new Exception("VIOLATION : " + msg); }

    // --- utilitaires numeriques ---
    public static double NextNormal(Random r)
    {
        double u1 = 1.0 - r.NextDouble();
        double u2 = 1.0 - r.NextDouble();
        return Math.Sqrt(-2.0 * Math.Log(u1)) * Math.Cos(2.0 * Math.PI * u2);
    }
    public static double LinRegSlope(double[] x, double[] y)
    {
        int n = x.Length;
        double mx = x.Average(), my = y.Average();
        double num = 0, den = 0;
        for (int i = 0; i < n; i++) { num += (x[i]-mx)*(y[i]-my); den += (x[i]-mx)*(x[i]-mx); }
        return num / den;
    }
    // HFD : antropy._higuchi_fd 0.1.6 (mne-features), copie declaree.
    public static double HiguchiFd(double[] x, int kmax)
    {
        int n = x.Length;
        double[] lx = new double[kmax]; double[] ly = new double[kmax];
        for (int ki = 0; ki < kmax; ki++)
        {
            int k = ki + 1; double m_lm = 0.0;
            for (int m = 0; m < k; m++)
            {
                double ll = 0.0;
                int n_max = (int)Math.Floor((n - m - 1) / (double)k);
                for (int j = 1; j < n_max; j++) ll += Math.Abs(x[m+j*k] - x[m+(j-1)*k]);
                ll /= k; ll *= (n - 1) / (k * (double)n_max); m_lm += ll;
            }
            m_lm /= k; lx[ki] = Math.Log(1.0 / k); ly[ki] = Math.Log(m_lm);
        }
        return LinRegSlope(lx, ly);
    }
    // LZC : antropy._lz_complexity 0.1.6 (Naereen), copie declaree.
    public static int LzComplexity(int[] s)
    {
        int n = s.Length;
        int complexity = 1, prefix_len = 1, len_substring = 1, max_len_substring = 1, pointer = 0;
        while (prefix_len + len_substring <= n)
        {
            if (s[pointer + len_substring - 1] == s[prefix_len + len_substring - 1])
                len_substring += 1;
            else
            {
                if (len_substring > max_len_substring) max_len_substring = len_substring;
                pointer += 1;
                if (pointer == prefix_len) { complexity += 1; prefix_len += max_len_substring; pointer = 0; max_len_substring = 1; }
                len_substring = 1;
            }
        }
        if (len_substring != 1) complexity += 1;
        return complexity;
    }
    public static double LzNorm(int[] s)
    {
        int n = s.Length;
        var seen = new HashSet<int>();
        foreach (var v in s) seen.Add(v);
        int b = seen.Count; if (b < 2) b = 2;
        return LzComplexity(s) / (n / Math.Log(n, b));
    }
    public static double Median(double[] x)
    {
        double[] s = (double[])x.Clone(); Array.Sort(s); int n = s.Length;
        return n % 2 == 1 ? s[n/2] : 0.5*(s[n/2 - 1] + s[n/2]);
    }

    // --- cohorte synthetique (AR(1) par region, effets sujet, effectifs inegaux) ---
    public class Cohort
    {
        public double[,] Feats; public int[] Y, Groups; public (int,int)[] Sizes;
        public int N => Y.Length;
    }
    public static Cohort SimulateCohort(double eta, int seed)
    {
        var rng = new Random(seed);
        double[] grad = new double[N_ROI];
        for (int r = 0; r < N_ROI; r++) grad[r] = N_ROI == 1 ? 0.0 : (double)r/(N_ROI-1);
        var hRows = new List<double[]>(); var lRows = new List<double[]>();
        var y = new List<int>(); var g = new List<int>(); var sizes = new List<(int,int)>();
        for (int sujet = 0; sujet < N_SUBJ; sujet++)
        {
            int n_lsd = rng.Next(4, 9); int n_pla = EPOCHS_PER_SUBJ - n_lsd;
            sizes.Add((n_pla, n_lsd));
            double s_rho = NextNormal(rng)*SIGMA_RHO_SUBJ;
            double s_hfd = NextNormal(rng)*SIGMA_HFD_SUBJ;
            double s_lzc = NextNormal(rng)*SIGMA_LZC_SUBJ;
            foreach (var (condition, n_epochs) in new[]{ (0, n_pla), (1, n_lsd) })
                for (int e = 0; e < n_epochs; e++)
                {
                    var rowH = new double[N_ROI]; var rowL = new double[N_ROI];
                    for (int r = 0; r < N_ROI; r++)
                    {
                        double rho = Math.Clamp(RHO_PLA - eta*condition*grad[r] + s_rho, 0.05, 0.97);
                        double[] x = new double[T_EPOCH]; x[0] = NextNormal(rng);
                        for (int t = 1; t < T_EPOCH; t++)
                            x[t] = rho*x[t-1] + Math.Sqrt(1.0 - rho*rho)*NextNormal(rng);
                        rowH[r] = HiguchiFd(x, KMAX_HFD) + s_hfd;
                        double med = Median(x);
                        int[] bin = new int[T_EPOCH];
                        for (int t = 0; t < T_EPOCH; t++) bin[t] = x[t] > med ? 1 : 0;
                        rowL[r] = LzNorm(bin) + s_lzc;
                    }
                    hRows.Add(rowH); lRows.Add(rowL); y.Add(condition); g.Add(sujet);
                }
        }
        int N = y.Count;
        var feats = new double[N, 2*N_ROI];
        for (int i = 0; i < N; i++)
            for (int r = 0; r < N_ROI; r++) { feats[i, r] = hRows[i][r]; feats[i, N_ROI+r] = lRows[i][r]; }
        return new Cohort { Feats = feats, Y = y.ToArray(), Groups = g.ToArray(), Sizes = sizes.ToArray() };
    }

    // --- foret aléatoire CART (BCL, Gini, bootstrap, importances d'impurete) ---
    public class Params { public int MaxDepth; public int MinLeaf; public Params(int d, int l){MaxDepth=d;MinLeaf=l;} }
    class CartNode { public int Feat = -1; public double Thr; public CartNode Left, Right; public int Pred; public double[] Imp; }
    class Cart
    {
        int d; double[,] X; int[] y; int maxDepth, minLeaf; Random rng;
        public CartNode Root;
        Cart(double[,] X, int[] y, int d, int maxDepth, int minLeaf, Random rng){this.X=X;this.y=y;this.d=d;this.maxDepth=maxDepth;this.minLeaf=minLeaf;this.rng=rng;}
        public static Cart Train(double[,] X, int[] y, int d, int[] idx, int maxDepth, int minLeaf, Random rng)
        { var t = new Cart(X,y,d,maxDepth,minLeaf,rng); t.Root = t.Recurse(idx, 0); return t; }
        CartNode Recurse(int[] idx, int depth)
        {
            int cnt = idx.Length;
            var node = new CartNode { Imp = new double[d] };
            int c1 = 0; for (int i = 0; i < cnt; i++) if (y[idx[i]]==1) c1++;
            int c0 = cnt - c1; int maj = c1 >= c0 ? 1 : 0;
            double gini = Gini(c0, c1, cnt);
            if (cnt < 2*minLeaf || (maxDepth >= 0 && depth >= maxDepth) || gini < 1e-12) { node.Pred = maj; return node; }
            double bestG = double.MaxValue; int bf = -1; double bth = 0;
            for (int f = 0; f < d; f++)
            {
                int[] order = idx.OrderBy(i => X[i, f]).ToArray();
                int nCand = Math.Min(16, cnt-1);
                bool[] cand = new bool[cnt];
                for (int c = 1; c <= nCand; c++) cand[(int)Math.Round((double)c*cnt/(nCand+1))] = true;
                int left0 = 0, left1 = 0;
                for (int pos = 1; pos < cnt; pos++)
                {
                    int cl = y[order[pos-1]]; if (cl==1) left1++; else left0++;
                    if (cand[pos])
                    {
                        double v0 = X[order[pos-1], f], v1 = X[order[pos], f];
                        if (v1 > v0)
                        {
                            double gL = Gini(left0, left1, pos);
                            double gR = Gini(c0-left0, c1-left1, cnt-pos);
                            double g = (double)pos/cnt*gL + (double)(cnt-pos)/cnt*gR;
                            if (g < bestG) { bestG = g; bf = f; bth = (v0+v1)/2; }
                        }
                    }
                }
            }
            if (bf < 0) { node.Pred = maj; return node; }
            var li = new List<int>(); var ri = new List<int>();
            for (int i = 0; i < cnt; i++) { if (X[idx[i], bf] < bth) li.Add(idx[i]); else ri.Add(idx[i]); }
            if (li.Count == 0 || ri.Count == 0) { node.Pred = maj; return node; }
            node.Feat = bf; node.Thr = bth;
            node.Left = Recurse(li.ToArray(), depth+1);
            node.Right = Recurse(ri.ToArray(), depth+1);
            int l1 = 0; foreach (var i in li) if (y[i]==1) l1++;
            int r1 = 0; foreach (var i in ri) if (y[i]==1) r1++;
            node.Imp[bf] += gini - (double)li.Count/cnt*Gini(li.Count-l1, l1, li.Count)
                                 - (double)ri.Count/cnt*Gini(ri.Count-r1, r1, ri.Count);
            return node;
        }
        double Gini(int c0, int c1, int n){ if (n<=0) return 0; double p0=(double)c0/n,p1=(double)c1/n; return 1-p0*p0-p1*p1; }
    }
    public class Forest
    {
        int nTrees, d; Random rng; int maxDepth, minLeaf; List<Cart> trees;
        public Forest(int nTrees, int d, int seed, int maxDepth, int minLeaf){this.nTrees=nTrees;this.d=d;this.rng=new Random(seed);this.maxDepth=maxDepth;this.minLeaf=minLeaf;}
        public void Fit(double[,] X, int[] y, int[] idx)
        {
            int cnt = idx.Length; trees = new List<Cart>();
            var boot = new int[cnt];
            for (int t = 0; t < nTrees; t++)
            {
                for (int i = 0; i < cnt; i++) boot[i] = idx[rng.Next(cnt)];
                trees.Add(Cart.Train(X, y, d, boot, maxDepth, minLeaf, rng));
            }
        }
        int Vote(double[] row)
        {
            int s0 = 0, s1 = 0;
            foreach (var t in trees)
            {
                CartNode n = t.Root;
                while (n.Feat >= 0) n = row[n.Feat] < n.Thr ? n.Left : n.Right;
                if (n.Pred == 1) s1++; else s0++;
            }
            return s1 >= s0 ? 1 : 0;
        }
        public double Score(double[,] X, int[] y, int[] idx)
        {
            int ok = 0;
            for (int i = 0; i < idx.Length; i++)
            {
                double[] row = new double[d];
                for (int f = 0; f < d; f++) row[f] = X[idx[i], f];
                if (Vote(row) == y[idx[i]]) ok++;
            }
            return (double)ok / idx.Length;
        }
        public double[] FeatureImportances()
        {
            var imp = new double[d];
            foreach (var t in trees) AddImp(t.Root, imp);
            double tot = 0; foreach (var v in imp) tot += v;
            if (tot > 0) for (int f = 0; f < d; f++) imp[f] /= tot;
            else for (int f = 0; f < d; f++) imp[f] = 1.0/d;
            return imp;
        }
        void AddImp(CartNode n, double[] acc)
        {
            if (n == null) return;
            if (n.Feat >= 0) { for (int f = 0; f < acc.Length; f++) acc[f] += n.Imp[f]; AddImp(n.Left, acc); AddImp(n.Right, acc); }
        }
    }

    // --- plis (groupes = sujet, stratifies = classe) ---
    public static List<(int[] tr, int[] te)> GroupedKfold(int n, int[] y, int[] groups, int k, int seed)
    {
        var rng = new Random(seed);
        var subjects = groups.Distinct().OrderBy(_ => rng.Next(int.MaxValue)).ToArray();
        var foldOfSubject = new Dictionary<int, int>();
        for (int i = 0; i < subjects.Length; i++) foldOfSubject[subjects[i]] = i % k;
        var folds = Enumerable.Range(0, k).Select(_ => new List<int>()).ToList();
        for (int i = 0; i < n; i++) folds[foldOfSubject[groups[i]]].Add(i);
        var res = new List<(int[], int[])>();
        for (int f = 0; f < k; f++)
        {
            var te = folds[f].ToArray();
            var tr = new List<int>();
            for (int g = 0; g < k; g++) if (g != f) tr.AddRange(folds[g]);
            res.Add((tr.ToArray(), te));
        }
        return res;
    }
    public static List<(int[] tr, int[] te)> StratifiedKfold(int n, int[] y, int k, int seed)
    {
        var rng = new Random(seed);
        var idx0 = Enumerable.Range(0, n).Where(i => y[i]==0).OrderBy(_ => rng.Next(int.MaxValue)).ToArray();
        var idx1 = Enumerable.Range(0, n).Where(i => y[i]==1).OrderBy(_ => rng.Next(int.MaxValue)).ToArray();
        var folds = Enumerable.Range(0, k).Select(_ => new List<int>()).ToList();
        SplitRound(idx0, folds, k); SplitRound(idx1, folds, k);
        var res = new List<(int[], int[])>();
        for (int f = 0; f < k; f++)
        {
            var te = folds[f].ToArray();
            var tr = new List<int>();
            for (int g = 0; g < k; g++) if (g != f) tr.AddRange(folds[g]);
            res.Add((tr.ToArray(), te));
        }
        return res;
    }
    static void SplitRound(int[] idx, List<List<int>> folds, int k)
    {
        for (int i = 0; i < idx.Length; i++) folds[i % k].Add(idx[i]);
    }

    // --- validation croisee imbriquee (exterieur 4, interieur 3) ---
    static (double score, Params p) GridSearch(double[,] X, int[] y, int[] groups, int[] trainIdx, int d, int seed, bool aware, Iit.Params[] grid, int innerFolds)
    {
        // plis internes construits sur le SOUS-ENSEMBLE d'entrainement (y/groups locaux)
        int m = trainIdx.Length;
        int[] localY = new int[m]; int[] localG = new int[m];
        for (int i = 0; i < m; i++) { localY[i] = y[trainIdx[i]]; localG[i] = groups[trainIdx[i]]; }
        double best = -1; Params bp = grid[0];
        foreach (var p in grid)
        {
            var folds = aware ? GroupedKfold(m, localY, localG, innerFolds, seed)
                              : StratifiedKfold(m, localY, innerFolds, seed);
            double s = 0;
            foreach (var (tr, te) in folds)
            {
                var gtr = tr.Select(l => trainIdx[l]).ToArray();
                var gte = te.Select(l => trainIdx[l]).ToArray();
                var f = new Forest(N_TREES, d, seed, p.MaxDepth, p.MinLeaf);
                f.Fit(X, y, gtr);
                s += f.Score(X, y, gte);
            }
            s /= folds.Count;
            if (s > best) { best = s; bp = p; }
        }
        return (best, bp);
    }
    public static double NestedCv(double[,] X, int[] y, int[] groups, int seed, bool aware, Iit.Params[] grid, int d)
    {
        var outer = aware ? GroupedKfold(X.GetLength(0), y, groups, 4, seed)
                          : StratifiedKfold(X.GetLength(0), y, 4, seed);
        double sum = 0;
        foreach (var (tr, te) in outer)
        {
            var (_, bp) = GridSearch(X, y, groups, tr, d, seed, aware, grid, 3);
            var f = new Forest(N_TREES, d, seed, bp.MaxDepth, bp.MinLeaf);
            f.Fit(X, y, tr);
            sum += f.Score(X, y, te);
        }
        return sum / outer.Count;
    }
    // raccourci fautif : grille choisie une fois sur toutes les donnees, puis CV groupee
    public static (double score, Params p) NonNestedCv(double[,] X, int[] y, int[] groups, int seed, Iit.Params[] grid, int d)
    {
        int n = X.GetLength(0);
        var all = Enumerable.Range(0, n).ToArray();
        var (_, bp) = GridSearch(X, y, groups, all, d, seed, true, grid, 3);
        var outer = GroupedKfold(n, y, groups, 4, seed);
        double sum = 0;
        foreach (var (tr, te) in outer)
        {
            var f = new Forest(N_TREES, d, seed, bp.MaxDepth, bp.MinLeaf);
            f.Fit(X, y, tr);
            sum += f.Score(X, y, te);
        }
        return (sum / outer.Count, bp);
    }
    // nulle par permutation des etiquettes : la procedure complete est rejouee
    public static (double observed, double q95, double p) PermutationNull(
        double[,] X, int[] y, int[] groups, int seed, bool aware, Iit.Params[] grid, int d, int nPerm)
    {
        double obs = NestedCv(X, y, groups, seed, aware, grid, d);
        var rng = new Random(seed + 1000);
        var nulls = new double[nPerm];
        var yp = (int[])y.Clone();
        for (int p = 0; p < nPerm; p++)
        {
            for (int i = yp.Length - 1; i > 0; i--)
            { int j = rng.Next(i + 1); int tmp = yp[i]; yp[i] = yp[j]; yp[j] = tmp; }
            nulls[p] = NestedCv(X, yp, groups, seed, aware, grid, d);
        }
        int ge = 0; foreach (var v in nulls) if (v >= obs) ge++;
        double pval = (1.0 + ge) / (1.0 + nPerm);
        Array.Sort(nulls);
        int qi = (int)Math.Ceiling(0.95 * nPerm) - 1;
        if (qi < 0) qi = 0; if (qi >= nPerm) qi = nPerm - 1;
        return (obs, nulls[qi], pval);
    }
}

// --- Mise en place des constantes ---
Iit.SEEDS = new[] { 0, 1, 7, 42 };
Iit.N_SUBJ = 17; Iit.N_ROI = 12; Iit.EPOCHS_PER_SUBJ = 12; Iit.T_EPOCH = 1200;
Iit.KMAX_HFD = 18; Iit.N_TREES = 100; Iit.N_PERM = 30;
Iit.RHO_PLA = 0.90; Iit.SIGMA_RHO_SUBJ = 0.05; Iit.SIGMA_HFD_SUBJ = 0.12; Iit.SIGMA_LZC_SUBJ = 0.035;
Console.WriteLine("C# " + Environment.Version + "  |  BCL (pas de NuGet, pas de reseau)");
Console.WriteLine("constantes : N_SUBJ " + Iit.N_SUBJ + " | T_EPOCH " + Iit.T_EPOCH
    + " | N_ROI " + Iit.N_ROI + " | N_TREES " + Iit.N_TREES
    + " | N_PERM " + Iit.N_PERM + " | SEEDS [" + string.Join(",", Iit.SEEDS) + "]");


The below script needs to be able to find the current output cell; this is an easy method to get it.

C# 10.0.15  |  BCL (pas de NuGet, pas de reseau)


constantes : N_SUBJ 17 | T_EPOCH 1200 | N_ROI 12 | N_TREES 100 | N_PERM 30 | SEEDS [0,1,7,42]


### Lecture du resultat

L'environnement est pose et les constantes sont imprimees : le terrain suit la convention de la serie (17 sujets simules comme dans le plan factoriel d'ICT-36g, 12 regions, 12 epochs par sujet, 1 200 points par epoch), et les graines de mesure sont celles de la serie (`0, 1, 7, 42`). Le nombre de permutations est ramene a 30 : le papier en utilise 1 000, ici le plancher de p atteignable est `1/31 = 0,032`, suffisant pour un seuil a 0,05 mais pas pour une p precise. C'est une limite declaree, pas un raccourci cache.


## 2. Le terrain synthetique

Chaque sujet porte un processus AR(1) par region, de persistance `rho_r`. La condition placebo garde `rho = 0,90` partout ; la condition LSD retire de la persistance selon un **gradient anterieur -> posterieur** `g_r` :

```
rho_r = clip(0.90 - eta * condition * g_r + s_i, 0.05, 0.97)
```

Trois ingredients rendent le terrain realiste et les questions Q2-Q3 mesurables :

- **`s_i`**, un decalage de persistance propre au sujet (ecart-type 0,05) ;
- des **effectifs par condition inegaux par sujet** (entre 4 et 8 epochs LSD sur 12) ;
- des **decalages de sujet sur les descripteurs** (HFD et LZC), qui rendent l'identite du sujet informativement riche quand les plis l'ignorent.

Les descripteurs sont ceux de la serie : `HiguchiFd(x, kmax=18)` et `LzNorm` sur le signal binarise a sa mediane.


In [2]:
// La cohorte de reference porte tous les constats descriptifs de la suite.
var C = Iit.SimulateCohort(0.06, Iit.SEEDS[0]);
// Trois vues des memes donnees : multi-region (Feats), HFD seul, LZC seul.
int n = C.N;
var X_MULTI = C.Feats;                                  // (n, 2*N_ROI)
var X_HFD = new double[n, Iit.N_ROI];
var X_LZC = new double[n, Iit.N_ROI];
for (int i = 0; i < n; i++)
    for (int r = 0; r < Iit.N_ROI; r++)
    { X_HFD[i, r] = C.Feats[i, r]; X_LZC[i, r] = C.Feats[i, Iit.N_ROI + r]; }
int nPla = 0, nLsd = 0;
for (int i = 0; i < n; i++) { if (C.Y[i] == 0) nPla++; else nLsd++; }
Console.WriteLine("epochs " + n + " | descripteurs multi " + (2 * Iit.N_ROI));
Console.WriteLine("placebo " + nPla + " | LSD " + nLsd);
Console.WriteLine("sujets " + C.Groups.Distinct().Count());


epochs 204 | descripteurs multi 24


placebo 107 | LSD 97


sujets 17


## 3. Le terrain, mesure

Avant de decoder, on regarde ce que le terrain contient : les effectifs par sujet, et l'ecart des deux familles de descripteurs entre conditions le long du gradient. Un terrain ou les familles ne bougent pas ne peut pas etre decode, et un terrain ou elles bougent partout ne teste rien de spatial.


In [3]:
// Effectifs par sujet (inegaux) + total.
Console.WriteLine("sujet   placebo   LSD");
int totPla = 0, totLsd = 0;
for (int s = 0; s < Iit.N_SUBJ; s++)
{
    Console.WriteLine(s + "        " + C.Sizes[s].Item1 + "      " + C.Sizes[s].Item2);
    totPla += C.Sizes[s].Item1; totLsd += C.Sizes[s].Item2;
}
Console.WriteLine();
Console.WriteLine("total placebo " + totPla + " | total LSD " + totLsd);

// Resume par descripteur et condition : la dispersion dit si l'effet depasse le bruit sujet.
void LigneResume(string nom, int idxDesc, int condition)
{
    int off = idxDesc == 0 ? 0 : Iit.N_ROI;   // 0 = HFD, 1 = LZC
    double sum = 0, sumsq = 0, sumPost = 0; int cnt = 0;
    for (int i = 0; i < n; i++)
        if (C.Y[i] == condition)
        {
            for (int r = 0; r < Iit.N_ROI; r++) { double v = C.Feats[i, off + r]; sum += v; sumsq += v * v; cnt++; }
            sumPost += C.Feats[i, off + (Iit.N_ROI - 1)];   // region 11 (posterieure)
        }
    double moy = sum / cnt;
    double sd = Math.Sqrt(Math.Max(0.0, sumsq / cnt - moy * moy));
    double post = sumPost / (cnt / Iit.N_ROI);
    Console.WriteLine(nom + "   " + (condition == 0 ? "placebo" : "LSD") + "   moy "
        + moy.ToString("0.###") + "   sd " + sd.ToString("0.###") + "   posterieur " + post.ToString("0.###"));
}
Console.WriteLine();
LigneResume("HFD", 0, 0); LigneResume("HFD", 0, 1);
LigneResume("LZC", 1, 0); LigneResume("LZC", 1, 1);


sujet   placebo   LSD


0        5      7


1        8      4


2        7      5


3        5      7


4        5      7


... (13 outputs left out)

HFD   placebo   moy 1,652   sd 0,104   posterieur 1,65


HFD   LSD   moy 1,694   sd 0,111   posterieur 1,727


LZC   placebo   moy 0,603   sd 0,115   posterieur 0,603


LZC   LSD   moy 0,671   sd 0,101   posterieur 0,716


### Lecture du resultat

Les effectifs par sujet sont inegaux (4 a 8 epochs LSD sur 12) alors que le total reste equilibre : c'est la structure qui rend la fuite de groupe mesurable en Q3. Les deux familles se separent dans le sens attendu — HFD et LZC montent sous LSD, et l'ecart se concentre a la region posterieure, ou le gradient vaut 1. La region anterieure, ou `g_r = 0`, ne porte aucun effet : c'est le temoin negatif interne du terrain.


## 4. Le decodeur : foret aleatoire sous validation croisee imbriquee

Deux methodes de la classe `Iit` portent tout le carnet :

- `NestedCv` : la procedure du papier. Pour chaque pli exterieur, une recherche d'hyperparametres en validation croisee **interne** (3 plis) sur le train seul, puis une evaluation unique sur le pli exterieur. Avec `aware=true`, les deux niveaux sont groupes par sujet et aucun sujet ne traverse la frontiere train/test ; avec `aware=false`, ce sont des plis stratifies qui ignorent le sujet.
- `NonNestedCv` : le raccourci fautif. La grille est choisie **une fois sur toutes les donnees**, puis reevaluee par validation croisee groupee : les hyperparametres ont vu les plis de test, la precision remonte d'autant.

Le pli exterieur reste a K = 4 et le pli interne a K = 3, comme dans le papier.


In [4]:
// Deux grilles : la grossiere (2 x 2 = 4 combinaisons) et la fine (4 x 4 = 16).
// MaxDepth = -1 signifie "sans limite" (l'equivalent de max_depth=None du jumeau Python).
var GRID_COARSE = new Iit.Params[] {
    new Iit.Params(-1, 1), new Iit.Params(-1, 4),
    new Iit.Params(8, 1),  new Iit.Params(8, 4),
};
var GRID_FINE = new Iit.Params[] {
    new Iit.Params(-1, 1), new Iit.Params(-1, 2), new Iit.Params(-1, 4), new Iit.Params(-1, 8),
    new Iit.Params(4, 1),  new Iit.Params(4, 2),  new Iit.Params(4, 4),  new Iit.Params(4, 8),
    new Iit.Params(8, 1),  new Iit.Params(8, 2),  new Iit.Params(8, 4),  new Iit.Params(8, 8),
    new Iit.Params(12, 1), new Iit.Params(12, 2), new Iit.Params(12, 4), new Iit.Params(12, 8),
};
Console.WriteLine("grilles posees : COARSE " + GRID_COARSE.Length + " combinaisons, FINE " + GRID_FINE.Length + " combinaisons");


grilles posees : COARSE 4 combinaisons, FINE 16 combinaisons


### Lecture du resultat — la structure des plis

Avant de mesurer une precision, on verifie que la validation fait ce qu'elle annonce : les sujets ne doivent pas traverser la frontiere train/test quand `aware=true`, et les deux classes doivent rester representees dans chaque pli de test.


In [5]:
// Structure des plis : les sujets ne doivent pas traverser la frontiere quand aware=true.
Console.WriteLine("groupes   pli   n test   pos test   sujets test   sujets train   sujets communs");
for (int aware = 0; aware < 2; aware++)
{
    var plis = aware == 1
        ? Iit.GroupedKfold(n, C.Y, C.Groups, 4, 0)
        : Iit.StratifiedKfold(n, C.Y, 4, 0);
    for (int k = 0; k < plis.Count; k++)
    {
        var (tr, te) = plis[k];
        int posTest = 0; for (int i = 0; i < te.Length; i++) posTest += C.Y[te[i]];
        double posRate = (double)posTest / te.Length;
        var setTr = new HashSet<int>(); foreach (var i in tr) setTr.Add(C.Groups[i]);
        var setTe = new HashSet<int>(); foreach (var i in te) setTe.Add(C.Groups[i]);
        int communs = setTe.Count(s => setTr.Contains(s));
        Console.WriteLine((aware == 1 ? "oui" : "non") + "      " + k + "     " + te.Length
            + "     " + posRate.ToString("0.##") + "      " + setTe.Count + "         " + setTr.Count + "        " + communs);
    }
}


groupes   pli   n test   pos test   sujets test   sujets train   sujets communs


non      0     52     0,48      16         17        16


non      1     51     0,47      17         17        17


non      2     51     0,47      17         17        17


non      3     50     0,48      17         17        17


oui      0     60     0,43      5         12        0


oui      1     48     0,48      4         13        0


oui      2     48     0,5      4         13        0


oui      3     48     0,5      4         13        0


### Lecture du resultat

Avec les plis groupes, la colonne « sujets communs » vaut zero partout : aucun sujet ne peut etre memorise par le modele avant d'etre teste. Sans groupement, chaque pli de test partage des sujets avec son train — c'est exactement le mecanisme de la fuite que Q3 chiffre. La colonne des positifs montre aussi que le desequilibre par sujet se propage aux plis : la stratification garde l'equilibre global, le groupement empeche la contamination.


## 5. Q1 — le decodeur recupere-t-il la condition ?

Mesure de la precision imbriquee pour les trois modeles du papier : mono-domaine HFD, mono-domaine LZC, multi-domaine (les deux familles empilees). Quatre graines de terrain. Puis la **nulle par permutation des etiquettes** : la procedure complete est rejouee `N_PERM` fois sur des etiquettes melangees. La p-value est `(1 + #{nulle >= observee}) / (1 + N_PERM)`.


In [6]:
// Precision imbriquee pour les 3 modeles du papier, 4 graines.
Console.WriteLine("modele    moyenne     sd      min     max");
double multiMean = 0;
foreach (var famX in new[] { ("HFD", X_HFD), ("LZC", X_LZC), ("multi", X_MULTI) })
{
    string fam = famX.Item1; double[,] X = famX.Item2;
    double[] accs = new double[Iit.SEEDS.Length];
    for (int s = 0; s < Iit.SEEDS.Length; s++)
        accs[s] = Iit.NestedCv(X, C.Y, C.Groups, Iit.SEEDS[s], true, GRID_COARSE, X.GetLength(1));
    double mean = accs.Average();
    double sd = Math.Sqrt(accs.Average(a => (a - mean) * (a - mean)));
    if (fam == "multi") multiMean = mean;
    Console.WriteLine(fam + "    " + mean.ToString("0.###") + "   " + sd.ToString("0.###")
        + "   " + accs.Min().ToString("0.###") + "   " + accs.Max().ToString("0.###"));
}


modele    moyenne     sd      min     max


HFD    0,789   0,026   0,747   0,816


LZC    0,787   0,023   0,75   0,809


multi    0,77   0,011   0,756   0,788


In [7]:
// Nulle par permutation des etiquettes : la procedure complete est rejouee N_PERM fois.
var (obsMulti, q95, pVal) = Iit.PermutationNull(X_MULTI, C.Y, C.Groups, Iit.SEEDS[0], true, GRID_COARSE, X_MULTI.GetLength(1), Iit.N_PERM);
Console.WriteLine("precision observee (multi, graine 0) " + obsMulti.ToString("0.###"));
Console.WriteLine("quantile 95 de la nulle " + q95.ToString("0.###") + "  |  p = " + pVal.ToString("0.###")
    + "  (plancher atteignable " + (1.0 / (1 + Iit.N_PERM)).ToString("0.###") + ")");
// Q1 : precision multi >= 0.65 ET observee au-dessus du quantile 95 de la nulle.
Iit.Check(multiMean >= 0.65 && obsMulti > q95, "Q1 : le decodeur ne depasse pas la nulle");
Console.WriteLine("Q1 : CONFIRMEE (multi " + multiMean.ToString("0.###") + " >= 0.65 ; observee "
    + obsMulti.ToString("0.###") + " > Q95 " + q95.ToString("0.###") + ")");


precision observee (multi, graine 0) 0,769


quantile 95 de la nulle 0,552  |  p = 0,032  (plancher atteignable 0,032)


Q1 : CONFIRMEE (multi 0,77 >= 0.65 ; observee 0,769 > Q95 0,552)


### Lecture du resultat — Q1

La prediction etait « precision multi-domaine >= 0,65 et au-dessus du quantile 95 de la nulle ». Le verdict se lit sur les deux nombres imprimes : la precision observee, et la position de la nulle. La largeur de la nulle est elle-meme instructive : c'est la variabilite qu'une precision de 0,5 produit par hasard sur 4 plis et 17 sujets, autrement dit la barre qu'une etude doit franchir pour parler de decodage. Une precision mono-domaine nettement plus basse que la multi-domaine signalerait que les deux familles portent une information partiellement complementaire.


## 6. Q2 — le biais du CV non imbrique

Le raccourci non imbrique choisit les hyperparametres sur l'ensemble des donnees, puis mesure par validation croisee groupee. Les plis de test ont influence le choix : la mesure est optimiste. La prediction est que le biais est positif, et qu'il **croit avec la finesse de la grille**.

Deux grilles sont comparees : la grossiere (4 combinaisons) et la fine (16 combinaisons). Pour chaque graine, le biais vaut `precision non imbriquee - precision imbriquee`.


In [8]:
// Biais du CV non imbrique : precision non imbriquee - precision imbriquee, par graine et grille.
Console.WriteLine("grille          graine   imbriquee   non imb.   biais   hyperparametres");
foreach (var ng in new[] { ("grossiere (4)", GRID_COARSE), ("fine (16)", GRID_FINE) })
{
    string nomGrille = ng.Item1; Iit.Params[] grille = ng.Item2;
    foreach (var seed in Iit.SEEDS)
    {
        double imb = Iit.NestedCv(X_MULTI, C.Y, C.Groups, seed, true, grille, X_MULTI.GetLength(1));
        var (rac, meilleurs) = Iit.NonNestedCv(X_MULTI, C.Y, C.Groups, seed, grille, X_MULTI.GetLength(1));
        Console.WriteLine(nomGrille + "     " + seed + "      " + imb.ToString("0.###")
            + "    " + rac.ToString("0.###") + "    " + (rac - imb).ToString("0.###")
            + "   d=" + meilleurs.MaxDepth + " ml=" + meilleurs.MinLeaf);
    }
}


grille          graine   imbriquee   non imb.   biais   hyperparametres


grossiere (4)     0      0,769    0,769    0   d=-1 ml=1


grossiere (4)     1      0,766    0,761    -0,004   d=-1 ml=4


grossiere (4)     7      0,788    0,782    -0,005   d=-1 ml=1


grossiere (4)     42      0,756    0,751    -0,005   d=-1 ml=1


fine (16)     0      0,773    0,763    -0,01   d=4 ml=2


fine (16)     1      0,766    0,756    -0,009   d=4 ml=4


fine (16)     7      0,793    0,782    -0,01   d=-1 ml=1


fine (16)     42      0,756    0,751    -0,005   d=-1 ml=1


### Lecture du resultat — Q2

La prediction etait `biais(fine) >= biais(grossiere) >= 0`, avec un seuil de materialite a 0,05. Deux issues sont informatives : si le biais reste nul sur les deux grilles, le raccourci ne coute pas en precision **ici** — mais il coute la garantie, parce que rien dans la procedure ne borne ce qu'il aurait pu valoir ; si la grille fine produit un biais materiel, la demonstration est complete. La colonne des hyperparametres choisis montre le mecanisme : une grille grossiere selectionne la meme configuration partout, une grille fine varie d'une graine a l'autre.


## 7. Q3 — la fuite de groupe

Meme procedure imbriquee, memes graines, meme grille : seule change la nature des plis. Les plis non groupes melangent les epochs sans respecter le sujet, si bien qu'un sujet peut etre present a la fois dans le train et dans le test. Comme le terrain donne a chaque sujet ses propres decalages de descripteurs et ses propres effectifs par condition, le modele peut apprendre la signature du sujet. Le papier utilise des plis groupes ; cette section mesure ce que coute leur absence.


In [9]:
// Fuite de groupe : meme procedure, seuls changent les plis (groupes vs non groupes).
Console.WriteLine("graine   plis groupes   plis non groupes   ecart");
double ecartSum = 0; int ecount = 0; double ecartMin = 1, ecartMax = 0;
foreach (var seed in Iit.SEEDS)
{
    double groupee = Iit.NestedCv(X_MULTI, C.Y, C.Groups, seed, true, GRID_COARSE, X_MULTI.GetLength(1));
    double fuite = Iit.NestedCv(X_MULTI, C.Y, C.Groups, seed, false, GRID_COARSE, X_MULTI.GetLength(1));
    double ecart = fuite - groupee;
    ecartSum += ecart; ecount++;
    ecartMin = Math.Min(ecartMin, ecart); ecartMax = Math.Max(ecartMax, ecart);
    Console.WriteLine(seed + "      " + groupee.ToString("0.###") + "      " + fuite.ToString("0.###") + "      " + ecart.ToString("0.###"));
}
double ecartMean = ecartSum / ecount;
Console.WriteLine();
Console.WriteLine("ecart moyen " + ecartMean.ToString("0.###") + "   min " + ecartMin.ToString("0.###") + "   max " + ecartMax.ToString("0.###"));
// Q3 : ecart (plis non groupes - plis groupes) >= +0.03.
Iit.Check(ecartMean >= 0.03, "Q3 : la fuite de groupe est inférieure au seuil");
Console.WriteLine("Q3 : CONFIRMEE (ecart moyen " + ecartMean.ToString("0.###") + " >= 0.03)");


graine   plis groupes   plis non groupes   ecart


0      0,769      0,853      0,085


1      0,766      0,829      0,063


7      0,788      0,833      0,046


42      0,756      0,824      0,067


ecart moyen 0,065   min 0,046   max 0,085


Q3 : CONFIRMEE (ecart moyen 0,065 >= 0.03)


### Lecture du resultat — Q3

La prediction etait un ecart d'au moins +0,03. L'ecart mesure est la part de la precision qui ne vient pas du signal mais de l'identite des sujets : elle mesure exactement ce qu'une etude rapporterait de trop en oubliant le groupement. Un ecart proche de zero aurait dit que ce terrain ne porte pas de signature de sujet exploitable. La comparaison des deux lignes de la section 4 est la meme mesure prise a l'envers : sans groupement, un decodeur parait meilleur qu'il n'est.


## 8. Q4 — le seuil de puissance

On balaie l'amplitude de l'effet injecte : `eta = 0` (aucune difference entre conditions), puis 0,02, 0,04 et 0,06. A chaque amplitude, deux graines de terrain et la precision multi-domaine imbriquee. La bande nulle mesuree en Q1 sert de reference.

L'amplitude `eta` s'exprime en retrait de persistance AR(1) au pole posterieur : `eta = 0,06` correspond a `rho` passant de 0,90 a 0,84.


In [10]:
// Balayage de l'amplitude : la precision doit retomber au hasard quand eta diminue.
double[] amplitudes = { 0.0, 0.02, 0.04, 0.06 };
Console.WriteLine("eta      precision moyenne   graine 0   graine 1");
double[] accsEta = new double[amplitudes.Length];
for (int i = 0; i < amplitudes.Length; i++)
{
    double eta = amplitudes[i];
    double[] accs = new double[2];
    for (int s = 0; s < 2; s++)
    {
        var ce = Iit.SimulateCohort(eta, Iit.SEEDS[s]);
        accs[s] = Iit.NestedCv(ce.Feats, ce.Y, ce.Groups, Iit.SEEDS[s], true, GRID_COARSE, ce.Feats.GetLength(1));
    }
    accsEta[i] = accs.Average();
    Console.WriteLine(eta + "   " + accs.Average().ToString("0.###") + "   " + accs[0].ToString("0.###") + "   " + accs[1].ToString("0.###"));
}
// Q4 : croissance monotone de la precision en eta.
bool mono = accsEta[0] <= accsEta[1] && accsEta[1] <= accsEta[2] && accsEta[2] <= accsEta[3];
Iit.Check(mono, "Q4 : la precision n'est pas monotone en eta");
Console.WriteLine("Q4 : CONFIRMEE (monotone " + string.Join(" -> ", accsEta.Select(a => a.ToString("0.###"))) + ")");


eta      precision moyenne   graine 0   graine 1


0   0,547   0,551   0,543


0,02   0,615   0,642   0,589


0,04   0,723   0,743   0,704


0,06   0,767   0,769   0,766


Q4 : CONFIRMEE (monotone 0,547 -> 0,615 -> 0,723 -> 0,767)


### Lecture du resultat — Q4

La prediction etait une croissance monotone et un franchissement de la bande nulle entre 0 et 0,06. Deux lectures : la valeur a `eta = 0` doit rester dans la bande nulle — si elle en sortait, l'instrument produirait une fausse detection sur un terrain sans effet, et tout le reste du carnet serait suspect ; et la pente entre 0 et 0,06 indique combien de sujets ou d'epochs il faudrait pour atteindre le seuil a une amplitude plus faible.


## 9. Ce que le decodeur regarde

Une precision ne dit pas **ou** le modele a trouve l'information. On entraine une foret sur toutes les donnees (pour la lecture seulement, pas pour une mesure de performance) et on lit ses importances d'impurete, par region et par famille. Le terrain a un temoin interne : la region 0, ou le gradient vaut zero, ne porte aucun effet injecte. Un decodeur qui attribue une importance forte a la region anterieure se trompe de source.

Le papier pousse ce geste plus loin, avec une **nulle d'importance** obtenue en permutant les etiquettes — c'est l'exercice 3.


In [11]:
// Lecture des importances : foret sur toutes les donnees (pas une mesure de performance).
var foret = new Iit.Forest(300, X_MULTI.GetLength(1), 0, -1, 1);
foret.Fit(X_MULTI, C.Y, Enumerable.Range(0, n).ToArray());
var imps = foret.FeatureImportances();
Console.WriteLine("region     HFD      LZC");
double hfdTot = 0, lzcTot = 0;
for (int r = 0; r < Iit.N_ROI; r++)
{
    double h = imps[r], l = imps[Iit.N_ROI + r];
    hfdTot += h; lzcTot += l;
    Console.WriteLine(r + "       " + h.ToString("0.###") + "    " + l.ToString("0.###"));
}
Console.WriteLine();
Console.WriteLine("importance totale HFD " + hfdTot.ToString("0.###") + "   |   LZC " + lzcTot.ToString("0.###"));
Console.WriteLine("importance region 0 (temoin) " + (imps[0] + imps[Iit.N_ROI]).ToString("0.###")
    + "   |   region posterieure 11 " + (imps[Iit.N_ROI - 1] + imps[2 * Iit.N_ROI - 1]).ToString("0.###"));
// Invariance de la zone d'egalite : la somme des importances vaut 1.
Iit.Check(Math.Abs(imps.Sum() - 1.0) < 1e-6, "somme des importances != 1");
Console.WriteLine("somme des importances = " + imps.Sum().ToString("0.#####") + "  (invariance verifiee)");


region     HFD      LZC


0       0,33    0,09


1       0,079    0,028


2       0,05    0,022


3       0,021    0,016


4       0,016    0,022


... (6 outputs left out)

11       0,022    0,037


importance totale HFD 0,636   |   LZC 0,364


importance region 0 (temoin) 0,42   |   region posterieure 11 0,058


somme des importances = 1  (invariance verifiee)


### Lecture du resultat

Les importances doivent pencher vers les regions posterieures, ou l'effet est injecte, et laisser la region 0 basse. L'importance d'impurete est cependant bruitee et biaisee en faveur des variables a cardinalite elevee : elle indique une tendance, pas une inference. C'est precisement pourquoi le papier mesure ses importances contre une nulle par permutation des etiquettes — la difference entre les deux gestes est l'objet de l'exercice 3.


## Exercices

Les trois exercices reutilisent l'instrument du carnet. Les cellules ci-dessous s'executent sans erreur : elles affichent une valeur placeholder tant que la consigne n'est pas implementee.


### Exercice 1 — decoder a une autre amplitude

Dans la cellule ci-dessous, generer une cohorte a l'amplitude `eta = 0.10` puis `eta = 0.20`, et calculer la precision multi-domaine en validation croisee imbriquee groupee pour chacune. Situer ces deux valeurs par rapport a la courbe de la section 8.

Indice : `Iit.SimulateCohort(eta, seed)` donne deja la vue multi dans `Feats` (2 x N_ROI) ; `Iit.NestedCv(..., aware: true, ...)` suffit.


In [12]:
// Exercice 1 : TODO etudiant
double result_eta = 0.0; // TODO etudiant
Console.WriteLine("eta = 0.10 -> " + result_eta);


eta = 0.10 -> 0


### Exercice 2 — la fuite selon l'amplitude

Pour chaque amplitude de `[0.0, 0.03, 0.06]`, mesurer l'ecart entre plis non groupes et plis groupes (Q3), et le consigner dans `ecarts` (dictionnaire amplitude -> ecart). Prediction a verifier : l'ecart de fuite est plus grand aux amplitudes faibles, parce que la part du signal y est plus petite.

Indice : `Iit.NestedCv(..., aware: false, ...)` moins `Iit.NestedCv(..., aware: true, ...)`, sur la meme cohorte.


In [13]:
// Exercice 2 : TODO etudiant
var ecarts = new Dictionary<double, double>(); // TODO etudiant
Console.WriteLine(string.Join("   ", ecarts.Select(kv => kv.Key + " -> " + kv.Value)));


### Exercice 3 — la nulle d'importance du papier

Entrainer une foret sur toutes les donnees, retenir l'importance de la region posterieure (11), puis repeter l'operation `n_perm` fois sur des etiquettes permutees. Consigner `(observee, nulle, p)` avec `p = (1 + #{nulle >= observee}) / (1 + n_perm)`. Le resultat attendu est une p petite pour la region posterieure, grande pour la region 0.

Indice : permuter `C.Y` (ex. reordonner les indices au hasard avec `rng`) detruit le lien signal-condition en gardant la distribution des descripteurs.


In [14]:
// Exercice 3 : TODO etudiant
double obs_imp = 0.0;
double[] nulle_imp = new double[0];
double p_imp = 1.0; // TODO etudiant
Console.WriteLine("observee " + obs_imp + "   nulle(n=" + nulle_imp.Length + ")   p " + p_imp);


observee 0   nulle(n=0)   p 1


## 10. Limites et suite

- **Nulle par permutation a 30 repetitions** au lieu des 1 000 du papier : le plancher de p est 0,032, la p ne se lit pas finement, et une nulle de queue est mal estimee.
- **La bande nulle de Q1 sert de reference a Q4** a toutes les amplitudes. La distribution des descripteurs change avec `eta`, donc la largeur exacte de la nulle pourrait varier legerement.
- **Importances d'impurete seulement**, pas de nulle d'importance : la tendance spatiale est lisible, l'inference ne l'est pas (exercice 3 pour le geste complet).
- **Deux familles de descripteurs** (HFD, LZC). Les familles spectrales du papier appartiennent a ICT-36f et ne sont pas decodees ici.
- **Le terrain est synthetique et unidimensionnel** : une seule manipulation (persistance AR(1)) sous-tend la difference de condition.
- **Aucune conclusion sur les donnees humaines** : le carnet calibre l'instrument, il ne replique pas le resultat du papier.
- **Jumeau C#** : la foret, les plis et la CV sont des reimplantations BCL — la procedure est la meme que le jumeau Python, mais les tirages `System.Random` et `numpy.default_rng` divergent, d'ou des precisions CV differentes a structure identique.

Suite possible : brancher le decodeur sur les traces MEG reelles du depot public lie a l'issue #20463, et comparer la precision obtenue a la bande nulle mesuree ici.


## Verdict

| Question | Prediction pre-enregistree | Mesure (jumeau C#) | Verdict |
|---|---|---|---|
| Q1 | recuperation au-dessus de la nulle | multi-domaine 0,770 ± 0,011 (HFD 0,789 ; LZC 0,787) ; observee 0,769 contre une nulle a quantile 95 = 0,552 ; p = 0,032, le plancher atteignable a 30 permutations | **CONFIRMEE** — le decodeur C# recupere la condition. Les deux familles mono (HFD 0,789, LZC 0,787) font a peu pres le meme ; le multi-domaine 0,770 est legerement en dessous, l'inverse du jumeau Python ou la LZC seule l'emportait |
| Q2 | biais >= 0, croissant avec la finesse de la grille | grille grossiere : biais moyen -0,003 (0, -0,004, -0,005, -0,005) ; grille fine : biais moyen -0,009 (-0,010, -0,009, -0,010, -0,005) | **INFIRMEE** — le biais est legerement negatif sur les deux grilles et ne croit pas avec la finesse, il reste dans le bruit des graines ; meme verdict que le jumeau Python |
| Q3 | ecart de fuite >= +0,03 | +0,065 en moyenne (min +0,046, max +0,085) ; 4 graines sur 4 au-dessus du seuil | **CONFIRMEE** — ignorer le sujet gonfle la precision de 5 a 8,5 points (jumeau Python : 0,082) |
| Q4 | croissance monotone, franchissement entre eta = 0 et 0,06 | 0,547 puis 0,615, 0,723, 0,767 pour eta = 0 / 0,02 / 0,04 / 0,06 ; quantile 95 de la nulle a 0,552 | **CONFIRMEE et precisee** — croissance monotone ; eta = 0 (0,547) reste dans la bande nulle (0,552) ; le franchissement se fait des eta = 0,02 (0,615 > 0,552) |

**Zone d'egalite exacte reproduite.** 204 echantillons, 17 sujets, placebo 107 / LSD 97, sujets communs 0 (plis groupes) et 16-17 (plis non groupes), somme des importances = 1, plancher de p = 1/(1+30) = 0,032. Les precisions CV different du jumeau Python (tirage `System.Random` vs `numpy.default_rng`) : ce n'est pas un ecart a combler, c'est la parite **semantique** qui fait foi.

**Ce que le jumeau C# etablit.** La procedure du papier — foret aleatoire, validation croisee imbriquee et groupee, nulle par permutation des etiquettes, fuite de groupe, seuil de puissance — est reimplantee sur la seule BCL (pas de NuGet, pas de reseau) et y reproduit les quatre verdicts qualitatifs du jumeau Python. Une particularite mesuree du jumeau C# : les importances d'impurete (section 9) sont dominees par la region 0 (0,42) au lieu de la region posterieure 11 (0,058). C'est le biais connu de l'importance d'impurete — qui credite le premier split d'un CART ecrit a la main et les variables a forte variance — exactement ce que la section 9 declare et ce que le geste de nulle par permutation (exercice 3) existe pour corriger.


## Pour aller plus loin

- Subramani, Nest et al. (2026), *LSD Reconfigures Cortical Dynamics Through Faster Brain Rhythms and Increased Fractal Dimension*, bioRxiv — section 2.10 (decodeur et validation croisee imbriquee).
- Issue [#20463](https://github.com/jsboige/CoursIA/issues/20463) — la distillation de ce papier dans la serie ICT.
- ICT-36c et ICT-36d — les deux descripteurs et leur reponse a la decorrelation temporelle.
- ICT-36g — la calibration du plan factoriel et du geste H4 sur le meme terrain AR(1).
- .NET BCL — `System.Random` (deterministe a graine egale sur un runtime donne) ; le jumeau Python s'appuie sur `numpy.default_rng`.
- [#15477](https://github.com/jsboige/CoursIA/issues/15477) — ICT-37, le probing lineaire held-out, autre facon de lire un etat latent.
